# Real - Compressão do JSON

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "last_expr_or_assign"

In [3]:
import io, zipfile, json, pprint

## Ver o que há dentro sem extrair

In [4]:
with zipfile.ZipFile("Comiset23_Real_Environment_Dataset.zip") as z:
    for info in z.infolist():
        print(info.filename, f"{info.file_size / 1e9:.2f}", "GB")

Comiset23_Real_Environment_Dataset.json 934.50 GB


In [5]:
with zipfile.ZipFile("Comiset23_Real_Environment_Dataset.zip") as z:
    print(f"{z.namelist()    = }")
    print(f"{z.namelist()[0] = }")

z.namelist()    = ['Comiset23_Real_Environment_Dataset.json']
z.namelist()[0] = 'Comiset23_Real_Environment_Dataset.json'


## Primeiro datapoint

In [6]:
with zipfile.ZipFile("Comiset23_Real_Environment_Dataset.zip") as z:
    # pra inspecionar tem que abrir
    with z.open(z.namelist()[0]) as f:
        # usar TextIOWrapper pra ler em string ao invés de 'bytes'
        with io.TextIOWrapper(f, encoding='utf-8') as text_file:
            first_line = text_file.readline()
            datapoint = json.loads(first_line)
            # por algum motivo isso ordena os campos, é bom desativar
            pprint.pprint(datapoint, sort_dicts=False)

{'_index': 'logs-endpoint-winevent-additional-2022.07.01',
 '_type': '_doc',
 '_id': '729e39cd5007321c1ad44ef30bb9785eb7ccd296',
 '_score': 1,
 '_source': {'Status': '3221226536',
             'z_elastic_ecs': {'agent': {},
                               'ecs': {'version': '8.0.0'},
                               'host': {},
                               'log': {},
                               'winlog': {'process': {'thread': {}}},
                               'event': {'kind': 'event',
                                         'code': '3033',
                                         'provider': 'Microsoft-Windows-CodeIntegrity',
                                         'action': 'CreateSection',
                                         'created': '2022-07-01T06:26:39.218Z'},
                               'user': {'type': 'Well Known Group',
                                        'domain': 'NT AUTHORITY',
                                        'name': 'Servicio de red',
        

In [7]:
type(datapoint)

dict

## DuckDB com configuração mínima

In [9]:
import pyarrow as pa, pyarrow.parquet as pq

In [10]:
import duckdb
from pathlib import Path

In [16]:
from ipyfilechooser import FileChooser
import os
# Initialize the FileChooser
fc = FileChooser()

FileChooser(path='C:\COMISET', filename='', title='', show_hidden=False, select_desc='Select', change_desc='Ch…

In [28]:
# Once selected, access the path via:
fc.selected

'E:\\Comiset23_Real_Environment_Dataset\\Comiset23_Real_Environment_Dataset.json'

In [29]:
REAL_ORIGINAL_JSON = Path(fc.selected)
REAL_PARQUET_OUT  = Path("Parquet/real_events.parquet")

WindowsPath('Parquet/real_events.parquet')

In [30]:
con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='14GB'")
# con.execute("PRAGMA temp_directory='/mnt/d/duckdb_tmp'") # caso precise de spill

## Compressão pra Parquet

In [31]:
con.execute(f"""
    COPY (
        SELECT * FROM read_ndjson(
            '{REAL_ORIGINAL_JSON}',
            auto_detect         = true,
            maximum_object_size = 104857600,
            ignore_errors       = true
        )
    )
    TO '{REAL_PARQUET_OUT}'
    (
        FORMAT         PARQUET,
        CODEC          'ZSTD',
        COMPRESSION_LEVEL 9
    )
""")

src  = Path(REAL_ORIGINAL_JSON).stat().st_size / 1e9
dest = Path(REAL_PARQUET_OUT).stat().st_size  / 1e9

print(f"JSON:    {src:.1f} GB")
print(f"Parquet: {dest:.1f} GB")
print(f"Razão:   {dest/src*100:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

JSON:    934.5 GB
Parquet: 13.3 GB
Razão:   1.4%


# Fim